# DocStruct - OHR-Bench leaderboard (Colab T4)

7 tools x 3,558 questions over 3,787 pages, under all three relevance modes.

**Before running:** Runtime -> Change runtime type -> **T4 GPU**.

## This notebook is resumable. Run every cell, top to bottom, every time.

Nothing recomputes what Drive already holds. Each cell checks Drive first and skips:

| Artifact | Cached at | Cost if lost |
|---|---|---|
| YOLO weights | `docstruct_bench/yolov8m-doclaynet.pt` | 50 MB download |
| Corpus zip + QA parquet | `docstruct_bench/ohrbench_download/` | 1.5 GB download |
| Detector proposals | `docstruct_bench/.bench_cache/` | the slow part of every run |
| Per-tool, per-mode progress | `.bench_cache/bench_ckpt_<tool>_<mode>.json` | hours |
| Finished leaderboards | `docstruct_bench/ohr_results_<mode>.json` | a whole mode |

If the session dies, re-run from the top. Section 3 prints exactly what was recovered.

## Relevance mode: all three

No rule is neutral to chunk size - `span` rewards large chunks, `page` rewards small
ones, `region` is built to be size-tolerant. Measured on this corpus
(`scripts/gold_reachability.py`): **80.2% of spans are reachable** under both `span` and
`region`, identically for every tool, so `span` is fair here and `page` has no special
claim. The paper reports all three and says whether the ranking survives. If DocStruct
wins only under `span`, that is uncomfortable and gets reported.

## On GPU utilisation

The card will look idle. Only YOLO layout detection (~20-40 ms/page) and the sentence
embeddings are on it; rasterisation, pdfplumber extraction, fusion and chunking are CPU
and dominate the ~0.6 s/page. A near-zero `nvidia-smi` is the workload's shape, not a bug.

## 1. GPU check

In [ ]:
!nvidia-smi
import torch, sys
if not torch.cuda.is_available():
    sys.exit('NO GPU. Runtime > Change runtime type > T4 GPU, then restart from cell 1.')
print('GPU ok:', torch.cuda.get_device_name(0))

## 2. Mount Drive

Everything expensive lands here. Free Colab reclaims sessions without warning; a
2-hour job that dies at minute 115 with its cache on local disk loses everything.

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

BENCH_DIR  = '/content/drive/MyDrive/docstruct_bench'
CACHE_DIR  = BENCH_DIR + '/.bench_cache'          # detector proposals + run checkpoints
DL_CACHE   = BENCH_DIR + '/ohrbench_download'     # 1.5GB pdfs.zip + QA parquet
DRIVE_W    = BENCH_DIR + '/yolov8m-doclaynet.pt'
for d in (BENCH_DIR, CACHE_DIR, DL_CACHE):
    os.makedirs(d, exist_ok=True)
print('bench dir ->', BENCH_DIR)

## 3. What is already cached

Read this before anything else - it says what this session will skip and what it will
actually spend time on. A tool showing all its documents done under a mode is finished;
that mode resumes instantly and only scores the remainder.

In [ ]:
import glob, json, os

ck = sorted(glob.glob(CACHE_DIR + '/bench_ckpt_*.json'))
print(f'--- run checkpoints ({len(ck)}) ---')
for f in ck:
    name = os.path.basename(f)[len('bench_ckpt_'):-len('.json')]
    try:
        d = json.load(open(f))
        print(f'  {name:<40} {len(d["done_docs"]):>5} docs, {len(d["per_question"]):>5} questions scored')
    except Exception as e:
        print(f'  {name:<40} UNREADABLE ({e}) - delete it and that tool/mode restarts')

print('--- finished leaderboards on Drive ---')
done_modes = [m for m in ('page','span','region') if os.path.exists(f'{BENCH_DIR}/ohr_results_{m}.json')]
print('  complete:', done_modes or 'none')
print('  to run  :', [m for m in ('page','span','region') if m not in done_modes])

print('--- other caches ---')
print('  weights :', 'yes' if os.path.exists(DRIVE_W) else 'no (will download 50MB)')
print('  corpus  :', [os.path.basename(p) for p in glob.glob(DL_CACHE + '/*')] or 'empty (will download ~1.5GB)')
print('  detector proposals:', len(glob.glob(CACHE_DIR + '/*')) - len(ck), 'files')

## 4. Clone

Idempotent: clones only if absent, otherwise fast-forwards. Safe to re-run after a
restart without losing the working tree.

In [ ]:
%cd /content
import os
if not os.path.isdir('/content/DocStruct/.git'):
    !git clone -b feat/paper-draft https://github.com/CandyButcher27/DocStruct
%cd /content/DocStruct
!git pull --ff-only
!git log --oneline -1

## 5. Install

`unstructured-inference` is explicit: `unstructured[pdf]` does not pull it, and
`partition_pdf` imports it at module load even under `strategy="fast"`.

**Pillow is pinned to the version Colab already imported.** Colab imports PIL before
your first cell runs. If pip upgrades Pillow on disk, this kernel keeps the old
`PIL._typing` in memory against the new modules, and the mismatch surfaces later as
`ImportError: cannot import name '_Ink' from 'PIL._typing'` the moment ultralytics loads
the model. Reinstalling cannot fix that from inside the same kernel - only a restart
can. Pinning means the on-disk Pillow never changes, so there is nothing to fix and no
restart is needed.

**Ignore pip's dependency-conflict warnings** about `jedi`, `requests`/`google-colab`
and `opentelemetry`/`google-adk`. Pip reports on the whole environment including Colab
preinstalls this benchmark never imports. Only a *traceback* matters.

In [ ]:
import PIL
PILLOW_PIN = PIL.__version__
print('holding pillow at', PILLOW_PIN)

!pip install -q -e ".[all,benchmark-heavy]" unstructured-inference pyarrow \
   llama-index-core llama-index-embeddings-huggingface "pillow=={PILLOW_PIN}"

## 6. Verify the install before spending two hours on it

Checks the things that actually break, not proxies for them. `get_adapters()` swallows
import errors and drops anything whose `available()` is False, so a missing dependency
would silently remove a tool from the leaderboard rather than fail. Read `MISSING`.

In [ ]:
import PIL, ultralytics
from ultralytics import YOLO      # the real consumer of Pillow; if PIL is broken, this raises
print('pillow', PIL.__version__, '| ultralytics', ultralytics.__version__, 'ok')

from docstruct.eval.adapters import get_adapters
TOOLS = ['docstruct','docstruct_geo','langchain','pymupdf4llm',
         'unstructured','llamaindex','llamaindex_semantic']
got = get_adapters(names=TOOLS, weights=None)
print('available:', sorted(got))
print('MISSING  :', sorted(set(TOOLS) - set(got)) or 'none')

## 7. Weights

Kept on Drive so a lost session costs a copy, not a download. docling is deliberately
out of the tool set (decided 2026-08-07) - it runs fine on Colab, but the paper carries
no docling row and must not imply one.

In [ ]:
import os, shutil
os.makedirs('weights', exist_ok=True)
W = 'weights/yolov8m-doclaynet.pt'

if not os.path.exists(DRIVE_W):
    print('not on Drive, downloading...')
    !wget -q --show-progress -O "{DRIVE_W}" https://huggingface.co/hantian/yolo-doclaynet/resolve/main/yolov8m-doclaynet.pt
if not os.path.exists(W):
    shutil.copy(DRIVE_W, W)      # local disk: the run reads this thousands of times
print(W, os.path.getsize(W) // 1024 // 1024, 'MB')

In [ ]:
# Confirm YOLO lands on the GPU. Expect cuda:0. Nothing in docstruct/ sets a device;
# ultralytics and sentence-transformers auto-select CUDA. If this says cpu, stop.
from docstruct.model.detector import ModelDetector
m = ModelDetector(weights=W)._ensure_model()
print('YOLO device:', next(m.model.parameters()).device)

## 8. Corpus (~1.6 GB)

`fetch_ohrbench.py` writes `data/ohrbench/*.pdf`, `data/qa/ohrbench.json` and
`reports/ohrbench_manifest.json`, and skips any file already present.

`data/ohrbench/_download` is symlinked to Drive, so a re-run after a lost session
re-extracts from the cached zip instead of re-downloading. The extracted PDFs stay on
local disk on purpose: the benchmark reads all 3,787 pages repeatedly and Drive's FUSE
mount is slower than re-extracting.

In [ ]:
import os
os.makedirs('data/ohrbench', exist_ok=True)
link = 'data/ohrbench/_download'
if os.path.isdir(link) and not os.path.islink(link):
    raise SystemExit(f'{link} is a real directory, not a link to Drive. Move it aside first.')
if not os.path.islink(link):
    os.symlink(DL_CACHE, link)
print('_download ->', os.path.realpath(link))

!python scripts/fetch_ohrbench.py

In [ ]:
import json, glob, collections
gold = json.load(open('data/qa/ohrbench.json'))
items = gold['items'] if isinstance(gold, dict) else gold
print(len(items), 'questions (expect 3558) |', len(glob.glob('data/ohrbench/*.pdf')), 'pdfs')
print('evidence_source:', dict(collections.Counter(i.get('evidence_source') for i in items)))
assert len(items) == 3558, 'gold is not the full corpus - do not start the run'

## 9. Optional: time-box the one risky tool (3 documents)

`llamaindex_semantic` embeds during chunking, so it is far slower than the other
splitters. Skip this cell if you have already run it once.

Read the per-document lines, not the exit code: the benchmark catches per-document
exceptions (`benchmark.py:239`), so a tool that fails on every document still exits 0
and reports an all-errors row. The 3 smoke docs are ~66 pages against 3,787 - **x57**.
Decision rule: 3/3 chunk -> keep it; 0/3 -> drop it and record the error verbatim;
1-2/3 -> drop it from the headline run, because the error rate is the finding.

In [ ]:
%%time
import glob, os, shutil
os.makedirs('/content/smoke3', exist_ok=True)
for p in sorted(glob.glob('data/ohrbench/*.pdf'))[:3]:
    shutil.copy(p, '/content/smoke3/')

!python -m docstruct.cli benchmark \
   --pdfs-dir /content/smoke3 --qa data/qa/ohrbench.json \
   --weights {W} \
   --tools llamaindex_semantic --relevance page \
   --report-md /content/smoke_lis.md --report-json /content/smoke_lis.json

## 10. The run - all three modes (~1.5-2 h each, cold)

Re-run this cell as many times as it takes. It skips any mode whose leaderboard is
already on Drive, and within a mode the checkpoint resumes per tool per document, so a
session that dies six tools in loses only the current document.

Checkpoints are keyed on tool **and** relevance mode (`benchmark.py:151`), so the modes
cannot inherit each other's numbers. There is no cheap re-scoring path - the checkpoint
stores scored results, not retrievals - so each mode is a full pass. The detector cache
in `CACHE_DIR` is shared across modes, and that is the expensive part, so the second and
third modes are meaningfully cheaper than the first.

Each mode is copied to Drive the moment it finishes, not at the end.

In [ ]:
import os, glob, shutil, subprocess, sys

TOOLS_ARG = 'docstruct,docstruct_geo,langchain,pymupdf4llm,unstructured,llamaindex,llamaindex_semantic'
MODES = ['page', 'span', 'region']
os.makedirs('reports', exist_ok=True)

for mode in MODES:
    md_name, js_name = f'ohr_report_{mode}.md', f'ohr_results_{mode}.json'
    if os.path.exists(f'{BENCH_DIR}/{js_name}'):
        print(f'== {mode}: already on Drive, restoring and skipping', flush=True)
        for f in (md_name, js_name):
            shutil.copy(f'{BENCH_DIR}/{f}', f'reports/{f}')
        continue

    print(f'== {mode}: running', flush=True)
    subprocess.run([
        sys.executable, '-m', 'docstruct.cli', 'benchmark',
        '--pdfs-dir', 'data/ohrbench', '--qa', 'data/qa/ohrbench.json',
        '--weights', W,
        '--tools', TOOLS_ARG,
        '--relevance', mode,
        '--cache-dir', CACHE_DIR,
        '--report-md', f'reports/{md_name}',
        '--report-json', f'reports/{js_name}',
    ], check=True)

    for f in (md_name, js_name):     # to Drive immediately, not after all three
        shutil.copy(f'reports/{f}', f'{BENCH_DIR}/{f}')
    print(f'== {mode}: saved to Drive', flush=True)

print('on Drive:', sorted(os.path.basename(p) for p in glob.glob(BENCH_DIR + '/ohr_results_*.json')))

## 11. Read the three leaderboards

In [ ]:
from IPython.display import Markdown, display
for mode in MODES:
    p = f'reports/ohr_report_{mode}.md'
    if os.path.exists(p):
        display(Markdown(f'### relevance = `{mode}`\n\n' + open(p).read()))

## 12. Download

Already on Drive; this is the second copy. Commit the JSONs to the repo locally - they
are the paper's evidence.

In [ ]:
!cd reports && zip -q -r /content/ohr_results.zip ohr_report_*.md ohr_results_*.json
from google.colab import files
files.download('/content/ohr_results.zip')

---
## Next, back on the local machine (no GPU needed)

Analysis on the JSONs, per `futureplans.md` section 3:

1. **Compare the three modes.** Does the ranking survive? The `page` run from 2026-08-07
   put DocStruct 5th of 6 at 0.6004 MRR against unstructured's 0.795, with the
   smallest-chunk tool winning - which is what `page` structurally rewards. Whether that
   is a property of DocStruct or of the metric is what these three runs decide.
2. **Slice by `evidence_source`** (text 2,666 / table 847 / equation 45). If DocStruct
   loses specifically on table or equation questions, that measurement justifies
   expanding the 5-label set. If it does not, do not expand it.
3. **Quantify the back-matter penalty.** DocStruct drops references by design, so under
   `page` relevance any question whose evidence page is in the back matter is
   structurally unreachable for us and reachable for everyone else. On the smoke,
   DocStruct's chunks stopped at page 11 of a 15-page paper while every other tool
   reached 14. Measure this before concluding anything from the page-mode numbers.

FinanceBench is a separate session (~15,000 pages, ~4x this run) and needs
`--relevance region`.